# TCN search: experiment report

Reads the run folders written by `python -m tcnsearch.search` (plus `tcnsearch.final` and `tcnsearch.bench` if run).
No training and no data access needed: rerunning this notebook regenerates every table and figure from the records.

* Run tables use validation **half A** (the development set every decision was made on).
* Half B (untouched) and the full validation set appear only in the finalists section.

In [ ]:
import sys; sys.path.insert(0, '..')   # repo root
from pathlib import Path
import json, pandas as pd
from IPython.display import Image, Markdown, display
from tcnsearch import report

RUNS = Path('../runs')          # change if your runs folder is elsewhere
OUT = Path('../report')
df = report.build(RUNS, OUT)
pd.set_option('display.max_rows', 300, 'display.width', 250)
print(len(df), 'finished runs')

## Decisions taken by the search

In [ ]:
dec = json.loads((RUNS / '_search' / 'decisions.json').read_text())
pd.DataFrame([{'decision': k, 'value': json.dumps(v['value']), 'rule': v['why']} for k, v in dec.items()])

## Missing or failed runs (reported, never silently dropped)

In [ ]:
f = RUNS / '_search' / 'missing_runs.json'
json.loads(f.read_text()) if f.exists() else []

## Ranking of the strongest configurations (half A)

In [ ]:
cols = ['phase','name','family','params','rf','lr','epochs','seed','A_scored','A_t0','A_t1','A_all','min_per_epoch','still_rising']
df.sort_values('A_scored', ascending=False)[cols].head(25)

## Phase results

In [ ]:
for f in ['phase2_kernel_rf.png', 'phase3_width.png', 'phase4_act_norm.png']:
    if (OUT / 'figures' / f).exists():
        display(Image(OUT / 'figures' / f))

## Training and validation curves, per phase

In [ ]:
for f in sorted((OUT / 'figures').glob('curves_phase*.png')):
    display(Markdown(f'**{f.stem}**')); display(Image(f))

## Finalists vs the verified GRU baseline (3 seeds, official metric, full validation and half B)

In [ ]:
f = RUNS / '_search' / 'final_table.csv'
pd.read_csv(f) if f.exists() else 'run: python -m tcnsearch.final'

In [ ]:
f = RUNS / '_search' / 'final_results.json'
if f.exists():
    R = json.loads(f.read_text())
    for s in R['finalists']:
        print(s['name'], '| TCN - GRU on B (scored):', s['vs_gru_B_scored'], '|', s['verdict_B_scored'])
        print('   blend with GRU:', s['blend_with_gru']['tcn_weight_chosen_on_A'], 'B', round(s['blend_with_gru']['B']['scored']['wp'], 4))

## CPU inference

In [ ]:
f = RUNS / '_search' / 'bench.json'
pd.DataFrame(json.loads(f.read_text())['runs']).T if f.exists() else 'run: python -m tcnsearch.bench (idle machine)'

## All runs

In [ ]:
df.drop(columns=['model_json'])

## Conclusions
*(written after the search: what improved performance, what did not, limitations, next investigations)*